# 🍔 Food Vision — Transfer Learning with EfficientNetB0
### 10-Class Food Classification using Feature Extraction & Fine-Tuning

---

## Project Overview
This project uses **Transfer Learning** to classify 10 categories of food images, leveraging a pre-trained **EfficientNetB0** model (trained on ImageNet with 1.2M+ images).  
Transfer learning allows us to achieve high accuracy with **only 10% of the training data** — a key advantage in real-world scenarios where labelled data is scarce.

**Dataset:** Food-101 (10-class subset, 10% training data)  
**Base Model:** EfficientNetB0 (pre-trained on ImageNet)  
**Task:** Multi-Class Image Classification (10 classes)  
**Strategy:** Feature Extraction → Fine-Tuning

### 10 Food Classes
```
chicken_curry | chicken_wings | fried_rice | grilled_salmon | hamburger
ice_cream     | pizza         | ramen      | steak          | sushi
```

### What is Transfer Learning?
Instead of training a CNN from scratch (which requires millions of images), we **reuse** a model already trained on a massive dataset.  
We freeze the base model's weights and only train a new classification head — then optionally **fine-tune** the top layers for our specific task.

```
ImageNet Weights (frozen)
        ↓
EfficientNetB0 Base  ←── Feature Extractor
        ↓
GlobalAveragePooling2D
        ↓
Dense(256, relu) + Dropout
        ↓
Dense(10, softmax)  ←── Our Custom Head
```

---

## Table of Contents
1. [Import Libraries](#1)
2. [Download & Prepare Dataset](#2)
3. [Explore the Dataset](#3)
4. [Build Data Pipeline](#4)
5. [Phase 1 — Feature Extraction](#5)
6. [Train Feature Extractor](#6)
7. [Phase 2 — Fine-Tuning](#7)
8. [Evaluate & Compare Phases](#8)
9. [Confusion Matrix & Per-Class Analysis](#9)
10. [Predict on Custom Images](#10)

## 1. Import Libraries

In [ ]:
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.applications import EfficientNetB0
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import seaborn as sns
import os
import zipfile
import urllib.request
import pathlib
import random
from sklearn.metrics import confusion_matrix, classification_report

print('TensorFlow version:', tf.__version__)
print('GPU available:', tf.config.list_physical_devices('GPU'))
tf.random.set_seed(42)

CLASS_NAMES = ['chicken_curry','chicken_wings','fried_rice','grilled_salmon',
               'hamburger','ice_cream','pizza','ramen','steak','sushi']

## 2. Download & Prepare Dataset

In [ ]:
# Download the 10-class, 10% training data subset
url      = 'https://storage.googleapis.com/ztm_tf_course/food_vision/10_food_classes_10_percent.zip'
filename = '10_food_classes_10_percent.zip'

print(f'Downloading {filename}...')
urllib.request.urlretrieve(url, filename)

print('Unzipping...')
with zipfile.ZipFile(filename, 'r') as zip_ref:
    zip_ref.extractall()

# Verify directory structure
print('\nDataset structure:')
for dirpath, dirnames, filenames in os.walk('10_food_classes_10_percent'):
    if filenames:
        print(f'  {len(filenames):>4} images in "{dirpath}"')

## 3. Explore the Dataset

In [ ]:
train_dir = pathlib.Path('10_food_classes_10_percent/train')
test_dir  = pathlib.Path('10_food_classes_10_percent/test')

print('Classes:', sorted([d.name for d in train_dir.iterdir()]))
print()
for cls in sorted([d.name for d in train_dir.iterdir()]):
    n_train = len(list((train_dir / cls).glob('*.jpg')))
    n_test  = len(list((test_dir  / cls).glob('*.jpg')))
    print(f'  {cls:<20} train: {n_train:>4}  |  test: {n_test:>4}')

In [ ]:
fig, axes = plt.subplots(2, 5, figsize=(16, 7))
axes = axes.ravel()

for i, cls in enumerate(CLASS_NAMES):
    folder   = train_dir / cls
    img_path = random.choice(list(folder.glob('*.jpg')))
    img      = mpimg.imread(img_path)
    axes[i].imshow(img)
    axes[i].set_title(cls.replace('_',' ').title(), fontsize=9)
    axes[i].axis('off')

plt.suptitle('One Sample per Food Class (10% training data)', fontsize=13)
plt.tight_layout()
plt.show()

## 4. Build Data Pipeline

In [ ]:
IMG_SIZE   = (224, 224)
BATCH_SIZE = 32
AUTOTUNE   = tf.data.AUTOTUNE

# EfficientNetB0 includes its own preprocessing — no manual rescaling needed
train_dataset = tf.keras.utils.image_dataset_from_directory(
    train_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=True,
    seed=42
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    test_dir,
    image_size=IMG_SIZE,
    batch_size=BATCH_SIZE,
    label_mode='int',
    shuffle=False,
    seed=42
)

# Data augmentation layer (only applied during training)
data_augmentation = keras.Sequential([
    layers.RandomFlip('horizontal'),
    layers.RandomRotation(0.15),
    layers.RandomZoom(0.15),
    layers.RandomHeight(0.15),
    layers.RandomWidth(0.15),
], name='data_augmentation')

# Optimize pipelines
train_dataset = train_dataset.map(
    lambda x, y: (data_augmentation(x, training=True), y),
    num_parallel_calls=AUTOTUNE
).prefetch(AUTOTUNE)

test_dataset = test_dataset.prefetch(AUTOTUNE)

print('Train batches:', len(train_dataset))
print('Test batches: ', len(test_dataset))

## 5. Phase 1 — Feature Extraction
> Freeze all EfficientNetB0 layers. Only train our custom classification head.
> This is fast and works well even with little data.

In [ ]:
def build_feature_extractor(num_classes=10):
    # Load EfficientNetB0 pre-trained on ImageNet — exclude top classifier
    base_model = EfficientNetB0(
        include_top=False,
        weights='imagenet',
        input_shape=(224, 224, 3)
    )
    # ── Freeze ALL base model layers ──────────────────────────
    base_model.trainable = False

    # ── Build custom head ─────────────────────────────────────
    inputs  = keras.Input(shape=(224, 224, 3), name='input_layer')
    x       = base_model(inputs, training=False)  # training=False keeps BN frozen
    x       = layers.GlobalAveragePooling2D(name='global_avg_pool')(x)
    x       = layers.Dense(256, activation='relu', name='dense_256')(x)
    x       = layers.Dropout(0.3, name='dropout')(x)
    outputs = layers.Dense(num_classes, activation='softmax', name='output')(x)

    model = keras.Model(inputs, outputs, name='EfficientNetB0_FeatureExtractor')
    return model, base_model

model, base_model = build_feature_extractor()

print(f'Total layers in base model: {len(base_model.layers)}')
print(f'Trainable layers: {sum(1 for l in model.layers if l.trainable)}')
print(f'Non-trainable layers: {sum(1 for l in model.layers if not l.trainable)}')

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

model.summary()

## 6. Train — Phase 1 (Feature Extraction)

In [ ]:
callbacks_phase1 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1),
    keras.callbacks.ModelCheckpoint(
        'phase1_feature_extraction.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1)
]

print('=== Phase 1: Feature Extraction ===')
print('Base model frozen — only training classification head\n')

history_phase1 = model.fit(
    train_dataset,
    epochs=10,
    validation_data=test_dataset,
    callbacks=callbacks_phase1,
    verbose=1
)

loss_p1, acc_p1 = model.evaluate(test_dataset, verbose=0)
print(f'\nPhase 1 Test Accuracy: {acc_p1*100:.2f}%')

## 7. Phase 2 — Fine-Tuning
> Unfreeze the **top layers** of EfficientNetB0 and continue training with a very low learning rate.  
> This adjusts the pre-trained features for our specific food classification task.

In [ ]:
# ── Unfreeze the top 20 layers of the base model ─────────────
base_model.trainable = True

# Freeze all layers EXCEPT the last 20
for layer in base_model.layers[:-20]:
    layer.trainable = False

trainable_count = sum(1 for l in base_model.layers if l.trainable)
print(f'Unfrozen layers in base model: {trainable_count} / {len(base_model.layers)}')

# ── Recompile with much lower learning rate ───────────────────
# Critical: use 10x lower LR than Phase 1 to avoid catastrophic forgetting
model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-4),  # 10x lower
    loss='sparse_categorical_crossentropy',
    metrics=['accuracy']
)

print('\nModel recompiled for fine-tuning!')
print('Learning rate: 0.0001 (10× lower than Phase 1)')

In [ ]:
callbacks_phase2 = [
    keras.callbacks.EarlyStopping(
        monitor='val_accuracy', patience=5,
        restore_best_weights=True, verbose=1),
    keras.callbacks.ReduceLROnPlateau(
        monitor='val_accuracy', factor=0.5,
        patience=3, min_lr=1e-7, verbose=1),
    keras.callbacks.ModelCheckpoint(
        'phase2_fine_tuned.keras',
        monitor='val_accuracy', save_best_only=True, verbose=1)
]

print('=== Phase 2: Fine-Tuning ===')
print('Top 20 EfficientNetB0 layers unfrozen\n')

INITIAL_EPOCH = len(history_phase1.history['accuracy'])  # continue epoch count

history_phase2 = model.fit(
    train_dataset,
    epochs=INITIAL_EPOCH + 10,
    initial_epoch=INITIAL_EPOCH,
    validation_data=test_dataset,
    callbacks=callbacks_phase2,
    verbose=1
)

loss_p2, acc_p2 = model.evaluate(test_dataset, verbose=0)
print(f'\nPhase 2 Test Accuracy: {acc_p2*100:.2f}%')
print(f'Improvement from fine-tuning: +{(acc_p2 - acc_p1)*100:.2f}%')

## 8. Evaluate & Compare Both Phases

In [ ]:
# Combine both history objects for a full training curve
def combine_histories(h1, h2):
    combined = {}
    for key in h1.history:
        combined[key] = h1.history[key] + h2.history[key]
    return combined

combined = combine_histories(history_phase1, history_phase2)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))
p1_end = len(history_phase1.history['accuracy'])

# Accuracy
ax1.plot(combined['accuracy'],     label='Train', color='steelblue')
ax1.plot(combined['val_accuracy'], label='Validation', color='orange')
ax1.axvline(p1_end - 1, color='gray', linestyle='--', alpha=0.7, label='Fine-tune starts')
ax1.set_title('Accuracy — Phase 1 → Phase 2')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Accuracy')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Loss
ax2.plot(combined['loss'],     label='Train', color='steelblue')
ax2.plot(combined['val_loss'], label='Validation', color='orange')
ax2.axvline(p1_end - 1, color='gray', linestyle='--', alpha=0.7, label='Fine-tune starts')
ax2.set_title('Loss — Phase 1 → Phase 2')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Loss')
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.suptitle('Transfer Learning Training Curve (EfficientNetB0)', fontsize=13)
plt.tight_layout()
plt.show()

print('\n=== Phase Comparison ===')
print(f'Phase 1 (Feature Extraction): {acc_p1*100:.2f}%')
print(f'Phase 2 (Fine-Tuning):        {acc_p2*100:.2f}%')
print(f'Improvement:                  +{(acc_p2-acc_p1)*100:.2f}%')

## 9. Confusion Matrix & Per-Class Analysis

In [ ]:
# Collect all predictions
y_pred_probs = model.predict(test_dataset, verbose=0)
y_pred       = np.argmax(y_pred_probs, axis=1)

# Collect true labels from the dataset
y_true = np.concatenate([y for x, y in test_dataset], axis=0)

print('=== Per-Class Classification Report ===')
print(classification_report(y_true, y_pred,
      target_names=[c.replace('_',' ').title() for c in CLASS_NAMES]))

# Confusion matrix
cm = confusion_matrix(y_true, y_pred)
display_names = [c.replace('_','\n').title() for c in CLASS_NAMES]

plt.figure(figsize=(13, 10))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=display_names,
            yticklabels=display_names)
plt.title('Confusion Matrix — Food Vision (EfficientNetB0 Fine-Tuned)', fontsize=13)
plt.ylabel('Actual')
plt.xlabel('Predicted')
plt.xticks(fontsize=8)
plt.yticks(fontsize=8, rotation=0)
plt.tight_layout()
plt.show()

In [ ]:
# Per-class accuracy bar chart
per_class_acc = cm.diagonal() / cm.sum(axis=1)

plt.figure(figsize=(12, 5))
bars = plt.bar(range(10), per_class_acc * 100,
               color=['green' if a >= 0.8 else 'orange' if a >= 0.6 else 'red'
                      for a in per_class_acc])
plt.xticks(range(10), [c.replace('_','\n').title() for c in CLASS_NAMES], fontsize=9)
plt.axhline(y=acc_p2*100, color='navy', linestyle='--', label=f'Overall: {acc_p2*100:.1f}%')
plt.ylabel('Accuracy (%)')
plt.title('Per-Class Accuracy — EfficientNetB0 Fine-Tuned')
plt.ylim(0, 105)
plt.legend()
for i, v in enumerate(per_class_acc):
    plt.text(i, v*100 + 1.5, f'{v*100:.0f}%', ha='center', fontsize=8)
plt.tight_layout()
plt.show()

## 10. Predict on Custom Images

In [ ]:
def predict_food(model, image_path, class_names=CLASS_NAMES, img_size=(224, 224)):
    """
    Predict the food class for a given image path.
    Returns predicted class name and confidence.
    """
    img = tf.keras.utils.load_img(image_path, target_size=img_size)
    img_array = tf.keras.utils.img_to_array(img)
    img_array = tf.expand_dims(img_array, axis=0)  # shape: (1, 224, 224, 3)

    pred_probs = model.predict(img_array, verbose=0)[0]
    pred_idx   = np.argmax(pred_probs)
    pred_class = class_names[pred_idx].replace('_', ' ').title()
    confidence = pred_probs[pred_idx]

    # Plot top-3 predictions
    top3_idx = np.argsort(pred_probs)[::-1][:3]

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
    ax1.imshow(tf.keras.utils.load_img(image_path))
    ax1.set_title(f'Prediction: {pred_class}\nConfidence: {confidence*100:.1f}%')
    ax1.axis('off')

    top3_names = [class_names[i].replace('_',' ').title() for i in top3_idx]
    top3_probs = [pred_probs[i]*100 for i in top3_idx]
    colors     = ['#2ecc71' if i == 0 else '#3498db' for i in range(3)]
    ax2.barh(top3_names[::-1], top3_probs[::-1], color=colors[::-1])
    ax2.set_xlabel('Confidence (%)')
    ax2.set_title('Top-3 Predictions')
    ax2.set_xlim(0, 100)

    plt.tight_layout()
    plt.show()
    return pred_class, confidence


# Predict on random test images from each class
print('=== Predictions on Random Test Images ===\n')
for cls in random.sample(CLASS_NAMES, 3):  # 3 random classes
    folder    = test_dir / cls
    img_path  = random.choice(list(folder.glob('*.jpg')))
    pred, conf = predict_food(model, str(img_path))
    true_label = cls.replace('_',' ').title()
    status = '✅' if pred == true_label else '❌'
    print(f'{status}  True: {true_label:<20} | Predicted: {pred:<20} ({conf*100:.1f}%)')

---

## Summary

| Phase | Strategy | Trainable Params | Accuracy |
|-------|----------|-----------------|----------|
| Phase 1 | Feature Extraction (base frozen) | ~330K | ~70–75% |
| Phase 2 | Fine-Tuning (top 20 layers unfrozen) | ~2.5M | ~80–85% |

### Key Takeaways
- **Transfer learning works with very little data** — only 10% of Food-101 training images were used
- **Feature extraction is a great starting point** — fast to train, solid baseline
- **Fine-tuning always adds improvement** — but requires a lower learning rate to avoid forgetting
- **EfficientNetB0 is efficient and powerful** — only 5.3M total parameters, yet very accurate

### Possible Extensions
- Use 100% of the training data and compare accuracy
- Try MobileNetV2 or ResNet50 as the base model
- Add test-time augmentation (TTA) for higher accuracy
- Deploy the model with TensorFlow Lite for mobile inference